In [33]:
import asyncio
from picoagents import Agent
from picoagents.llm import OpenAIChatCompletionClient
import os
from dotenv import load_dotenv

In [34]:
# Setup the language model client
load_dotenv("/home/hemamgholizadeh/multiagents/picoagent/var.env")

client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY")
)


In [35]:
poet = Agent(
    name="poet",
    description="Haiku poet. ",
    instructions="You are a haiku poet. ",
    model_client=client
)

In [36]:
async def test_poet():
    response = await poet.run("Write a haiku about cherry bloosoms in spring")
    print(f"Poet says: {response}")

await test_poet()

Poet says: [user] 12:35:08 | Write a haiku about cherry bloosoms in spring
[poet] 12:35:09 | Cherry blooms whisper,  
Petals dance on gentle winds,  
Spring's soft breath awakes.

[usage] duration: 1.1s, tokens: in:30, out:20 | finish: stop


In [37]:
critic = Agent(
    name="critic",
    description="Poetry critic who provides constructive feedback on haikus.",
    instructions="You are a hauku critic. \
        suggestions for environment. Be constructive and brief. \
            If satisfied with the hauku, respnd with 'APPROVED'",
    model_client=client
)

In [38]:
async def test_critic():
    haiku = ("Petals drift like snow,  \
        Whispers of spring fill the air—  \
        Pink clouds grace the sky.")
    response = await critic.run(f"Please critique this haiku: {haiku}")
    print(f"Critic says: {response}")
await test_critic()

Critic says: [user] 12:35:09 | Please critique this haiku: Petals drift like snow,          Whispers of spring fill the air—          Pink clouds grace the sky.
[critic] 12:35:13 | This haiku beautifully evokes a sense of transformation and renewal. The imagery of petals drifting like snow is striking and creates a serene visual. However, consider tightening the focus on either the sound of spring or the visual elements; blending both can dilute the impact. Perhaps you could emphasize the auditory sensations of spring more distinctly. Overall, a lovely piece!

REWRITE suggestion:  
Petals drift softly,  
Whispers of spring fill the air—  
Pink clouds grace the sky.

If adjusted for clarity and focus, it could be even stronger. 

APPROVED

[usage] duration: 3.5s, tokens: in:72, out:112 | finish: stop


In [39]:
from picoagents.orchestration import RoundRobinOrchestrator
from picoagents.termination import MaxMessageTermination, TextMentionTermination

# create termination conditions
termination = (MaxMessageTermination(max_messages=4) |
               TextMentionTermination(text="APPROVED"))

orchestrator = RoundRobinOrchestrator(agents=[poet, critic],
                                      termination=termination,
                                      max_iterations=4)

In [40]:
# Run orchestration

async def run_orchestration():
    task="Write a haiku about cherry blossoms in spring"
    stream = orchestrator.run_stream(task)
    async for message in stream:
        print(f"{message}")
await run_orchestration()

[user] 12:35:13 | Write a haiku about cherry blossoms in spring
[poet] 12:35:14 | Petals gently fall,  
Soft whispers of spring awake,  
Nature's fleeting blush.
[critic] 12:35:16 | The imagery is lovely, capturing the essence of cherry blossoms well. To enhance the emotional connection, consider a slight tweak to deepen the sense of ephemerality and beauty.

Suggestion:  
Focus on a sensory detail that paints a picture of spring and evokes a more personal response.

APPROVED
🏁 RoundRobinOrchestrator: The imagery is lovely, capturing the essence of cherry blossoms well. To enhance... | duration: 2.6s, tokens: in:184, out:76, calls: 2 . Stop reason: Composite (any): Text mention found: 'APPROVED'


In [42]:
from threading import Thread
from picoagents.webui import serve

def start_webui():
    serve(entities=[orchestrator], port=8070, auto_open=False)


print("PicoAgents WebUI: http://127.0.0.1:8070")

PicoAgents WebUI: http://127.0.0.1:8070


INFO:     127.0.0.1:55420 - "GET /api/entities HTTP/1.1" 200 OK
INFO:     127.0.0.1:55420 - "GET /api/sessions?entity_id=RoundRobinOrchestrator HTTP/1.1" 200 OK
INFO:     127.0.0.1:55420 - "GET /api/sessions?entity_id=RoundRobinOrchestrator HTTP/1.1" 200 OK
INFO:     127.0.0.1:55408 - "GET /api/sessions/980f6973-5a54-48c5-a41f-449c8cdb20e7/messages HTTP/1.1" 200 OK
